In [ ]:
from huggingface_hub import login

login(token='...')

In [ ]:
#DOWNLOADED THE FOLDERS WITH THIS
'''from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='computage/computage_bench', 
    repo_type="dataset",
    local_dir='.')
'''

In [ ]:
import pandas as pd
import numpy as np

splits = {
    "benchmark": "computage_bench_meta.tsv",
    "train": "computage_train_meta.tsv"
}

# Load both meta files
df_meta_benchmark = pd.read_csv("hf://datasets/computage/computage_bench/" + splits["benchmark"], sep="\t", index_col=0)
df_meta_train = pd.read_csv("hf://datasets/computage/computage_bench/" + splits["train"], sep="\t", index_col=0)

In [ ]:
#function for filling missing values with mean
def fill_missing_with_mean(df):
    return df.fillna(df.mean())

In [25]:
def detect_normalisation(df):
    """
    Detects if numeric values are mostly Beta-values (in range [0,1]) 
    and rounds values close to 0 or 1.
    
    Parameters:
    df (pd.DataFrame): Input DataFrame containing methylation values.
    
    Returns:
    pd.DataFrame: Transformed DataFrame with Beta-values rounded where necessary.
    """
    numeric_df = df.select_dtypes(include=["number"])  # Select numeric columns

    # Check if at least 99% of values are in [0,1] range
    in_range_mask = numeric_df.applymap(lambda x: 0 <= x <= 1)
    percent_in_range = in_range_mask.mean().mean()

    if percent_in_range >= 0.99:
        # Round values close to 0 or 1
        rounded_df = numeric_df.copy()
        rounded_df[rounded_df > 0.99] = 1
        rounded_df[rounded_df < 0.01] = 0
        return rounded_df  # Return cleaned Beta-values
    else:
        raise ValueError("Data does not appear to be Beta-values. Possible M-values, Logit(β), or Percentage Methylation.")

In [ ]:
import pandas as pd
import glob
import os

def process_and_merge_parquet_files(folder_path):
    """
    Reads all .parquet files from the specified folder, processes them, and merges them.

    Processing includes:
    - Converting column names to lowercase (only letters and numbers).
    - Filling missing values using `fill_missing_with_mean`.
    - Applying `detect_normalisation` function.
    - Merging DataFrames side by side.

    Parameters:
    folder_path (str): Path to the folder containing .parquet files.

    Returns:
    pd.DataFrame: The merged DataFrame.
    """
    
    # Get all .parquet files in the folder
    parquet_files = glob.glob(os.path.join(folder_path, "*.parquet"))

    processed_dfs = []

    for file in parquet_files:
        df = pd.read_parquet(file)

        # Convert column names to lowercase (only letters and numbers)
        df.columns = [col.lower() for col in df.columns]

        # Apply transformations
        df = fill_missing_with_mean(df)
        df = detect_normalisation(df)

        # Store the processed DataFrame
        processed_dfs.append(df)

    # Merge all DataFrames side by side
    result = pd.concat(processed_dfs, axis=1)

    return result

#merge the train parquet files
train_betas = process_and_merge_parquet_files("./data/train/")
#save the merged DataFrame to a CSV file
train_betas.to_csv('train_betas.csv', index=False)

/tmp/ipykernel_3916097/2601712886.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  in_range_mask = numeric_df.applymap(lambda x: 0 <= x <= 1)
/tmp/ipykernel_3916097/2601712886.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  in_range_mask = numeric_df.applymap(lambda x: 0 <= x <= 1)
/tmp/ipykernel_3916097/2601712886.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  in_range_mask = numeric_df.applymap(lambda x: 0 <= x <= 1)
/tmp/ipykernel_3916097/2601712886.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  in_range_mask = numeric_df.applymap(lambda x: 0 <= x <= 1)
/tmp/ipykernel_3916097/2601712886.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  in_range_mask = numeric_df.applymap(lambda x: 0 <= x <= 1)
/tmp/ipykernel_3916097/2601712886.py:15: FutureWarning: DataFrame.applymap has b

In [31]:
train_betas

,gsm1244499,gsm1244500,gsm1244501,gsm1244502,gsm1244503,gsm1244504,gsm1244505,gsm1244506,gsm1244507,gsm1244508,...,gsm4599899,gsm4599902,gsm4599903,gsm4599904,gsm4599906,gsm4599908,gsm4599909,gsm4599910,gsm4599911,gsm4599912
index,,,,,,,,,,,,,,,,,,,,,
cg00000029,0.611614,0.650165,0.556393,0.560802,0.453518,0.468323,0.472464,0.347559,0.418118,0.362770,...,0.435340,0.387643,0.416060,0.379702,0.377461,0.392739,0.417564,0.436442,0.440008,0.432328
cg00000165,0.255215,0.246081,0.349616,0.247382,0.362221,0.278715,0.251420,0.224335,0.245356,0.276730,...,0.142069,0.119381,0.149700,0.169790,0.124252,0.146462,0.150524,0.132626,0.159571,0.125079
cg00000236,0.728090,0.792496,0.805310,0.841228,0.733862,0.741599,0.819533,0.788315,0.783302,0.823247,...,0.631624,0.633163,0.651792,0.684932,0.656465,0.654223,0.622415,0.657265,0.650829,0.559303
cg00000289,0.720119,0.734806,0.700888,0.702669,0.728852,0.667953,0.651028,0.613968,0.648780,0.714638,...,0.716215,0.794667,0.657097,0.753737,0.721374,0.697518,0.750483,0.839342,0.759129,0.797669
cg00000321,0.304455,0.268263,0.323561,0.282914,0.295456,0.355035,0.252310,0.250897,0.238689,0.266487,...,0.198112,0.286586,0.236783,0.239489,0.181089,0.273595,0.269721,0.234263,0.268247,0.229820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ctl_Specificity_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ctl_Specificity_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ctl_Specificity_3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
